In [2]:
"""
KisanSaathi — Weather Module (No-Tool State Machine)
=====================================================
FLOW:
  IDLE → farmer asks weather → LLM extracts days → Python fetches lat/lon from DB
       → Python calls Open-Meteo API → returns formatted Hinglish result
       
LLM is used ONLY to extract "how many days" from farmer's message.
ALL data fetching is done directly by Python.
"""

import os
import asyncpg
import httpx
import json
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

# ─────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────
SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
OPEN_METEO_URL    = "https://api.open-meteo.com/v1/forecast"
DEFAULT_DAYS      = 3   # If farmer doesn't specify, use 3

# ─────────────────────────────────────────────
#  LLM — ONLY for extracting "days" from query
# ─────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROK_API_KEY"),
)


# ─────────────────────────────────────────────
#  LLM HELPER — Extract number of forecast days
# ─────────────────────────────────────────────
async def extract_weather_intent(user_msg: str) -> dict:
    """
    Ask LLM:
      1. Is this a weather query?
      2. How many days does the farmer want?
    Returns: {"is_weather_query": bool, "days": int}
    
    LLM does NOT fetch any data — only reads the farmer's text.
    """
    prompt = f"""Analyze this farmer's message. Respond ONLY with a JSON object.

Message: "{user_msg}"

Rules:
- is_weather_query = true if farmer is asking about weather/mausam/barish/temp/baarish/aandhi
- days = number of days requested (look for: "aaj"=1, "kal"=2, "teen din"=3, "ek hafte"=7)
- If no days mentioned, set days = 0 (means use default 3)
- Common keywords: mausam, barish, baarish, garmi, sardi, aandhi, temperature, weather, forecast

Respond ONLY with JSON:
{{"is_weather_query": true/false, "days": 0}}"""

    response = await llm.ainvoke([HumanMessage(content=prompt)])

    try:
        text = response.content.strip().replace("```json", "").replace("```", "").strip()
        result = json.loads(text)
        # Validate days range (1–16 is Open-Meteo max)
        days = int(result.get("days", 0))
        if days <= 0 or days > 16:
            days = DEFAULT_DAYS
        return {"is_weather_query": bool(result.get("is_weather_query")), "days": days}
    except Exception:
        # Fallback: keyword check
        keywords = ["mausam", "barish", "baarish", "weather", "garmi", "sardi", "aandhi", "temperature"]
        is_weather = any(k in user_msg.lower() for k in keywords)
        return {"is_weather_query": is_weather, "days": DEFAULT_DAYS}


# ─────────────────────────────────────────────
#  STEP 1 — Fetch lat/lon from DB using farmer_id
#  (Python calls DB directly — LLM not involved)
# ─────────────────────────────────────────────
async def get_farmer_lat_lon(farmer_id: str) -> tuple[float, float, str]:
    """
    Fetches center_lat, center_lon, field_name from farm_fields table.
    Returns the FIRST field associated with the farmer.
    Returns: (lat, lon, field_name)
    """
    conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
    try:
        query = """
            SELECT field_name, city_name, center_lat, center_lon
            FROM farm_fields
            WHERE farmer_id = $1
            LIMIT 1
        """
        row = await conn.fetchrow(query, farmer_id)
    finally:
        await conn.close()

    if not row:
        raise ValueError(f"No fields found for farmer_id: {farmer_id}")

    return float(row["center_lat"]), float(row["center_lon"]), row["field_name"], row["city_name"]


# ─────────────────────────────────────────────
#  STEP 2 — Call Open-Meteo API
#  (Python calls API directly — LLM not involved)
# ─────────────────────────────────────────────
async def fetch_weather_from_api(lat: float, lon: float, days: int) -> dict:
    """
    Calls Open-Meteo free API. No API key needed.
    Returns raw daily data dict.
    """
    params = {
        "latitude":      lat,
        "longitude":     lon,
        "daily":         "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
        "forecast_days": days,
        "timezone":      "Asia/Kolkata",   # IST — correct for Indian farmers
    }

    async with httpx.AsyncClient() as client:
        resp = await client.get(OPEN_METEO_URL, params=params, timeout=15.0)

    if resp.status_code != 200:
        raise ConnectionError(f"Open-Meteo API error: {resp.status_code}")

    return resp.json()


# ─────────────────────────────────────────────
#  STEP 3 — Format API response into Hinglish
#  (Pure Python formatting — LLM not involved)
# ─────────────────────────────────────────────
def format_weather_response(data: dict, field_name: str, city: str, days: int) -> str:
    """
    Converts raw Open-Meteo JSON into a clean Hinglish farmer-friendly message.
    """
    daily  = data.get("daily", {})
    dates  = daily.get("time", [])
    t_max  = daily.get("temperature_2m_max", [])
    t_min  = daily.get("temperature_2m_min", [])
    rain   = daily.get("precipitation_sum", [])
    wind   = daily.get("windspeed_10m_max", [])

    if not dates:
        return "⚠️  Mausam data abhi available nahi hai. Thodi der baad try karein."

    # Header
    lines = [
        f"🌤️  Mausam Forecast — {field_name} ({city})",
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
    ]

    for i in range(min(days, len(dates))):
        # Rain advisory
        rain_mm = rain[i] if rain[i] is not None else 0.0
        if rain_mm >= 10:
            rain_msg = f"🌧️  Baarish: {rain_mm}mm (Khet mein kaam band rakhein!)"
        elif rain_mm > 0:
            rain_msg = f"🌦️  Halki Baarish: {rain_mm}mm"
        else:
            rain_msg = "☀️  Baarish nahi"

        # Wind advisory
        wind_speed = wind[i] if wind[i] is not None else 0.0
        wind_msg = f"💨 Hawa: {wind_speed} km/h"
        if wind_speed > 40:
            wind_msg += " ⚠️  (Tej aandhi — fasal dhak lein!)"

        lines.append(
            f"\n📅 {dates[i]}\n"
            f"   🌡️  Temp: {t_min[i]}°C – {t_max[i]}°C\n"
            f"   {rain_msg}\n"
            f"   {wind_msg}"
        )

    lines.append("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    return "\n".join(lines)


# ─────────────────────────────────────────────
#  MAIN HANDLER — called from your state machine
# ─────────────────────────────────────────────
async def handle_weather_query(user_msg: str, farmer_id: str = SESSION_FARMER_ID) -> str | None:
    """
    Entry point. Returns:
      - Formatted weather string  → if it IS a weather query
      - None                      → if it is NOT a weather query (pass to next handler)

    FLOW:
      LLM detects intent + extracts days  (NLP only)
      Python fetches lat/lon from DB      (asyncpg)
      Python calls Open-Meteo API         (httpx)
      Python formats Hinglish response    (string formatting)
    """

    # ── STEP 0: LLM checks if this is even a weather question ───────
    intent = await extract_weather_intent(user_msg)

    if not intent["is_weather_query"]:
        return None   # Not a weather query — caller handles it

    days = intent["days"]

    # ── STEP 1: Python fetches location from DB (no LLM) ────────────
    try:
        lat, lon, field_name, city = await get_farmer_lat_lon(farmer_id)
    except ValueError as e:
        return f"⚠️  Aapka field data nahi mila: {e}\nPehle apna khet register karein."
    except Exception as e:
        return f"⚠️  Database se connect nahi ho saka: {e}"

    # ── STEP 2: Python calls Open-Meteo API (no LLM) ────────────────
    try:
        weather_data = await fetch_weather_from_api(lat, lon, days)
    except ConnectionError as e:
        return f"⚠️  Mausam server se data nahi aaya: {e}"
    except Exception as e:
        return f"⚠️  Unexpected error: {e}"

    # ── STEP 3: Python formats the response (no LLM) ────────────────
    return format_weather_response(weather_data, field_name, city, days)


# ─────────────────────────────────────────────
#  QUICK TEST  (Run directly in Jupyter)
# ─────────────────────────────────────────────
async def test_weather():
    test_queries = [
        "Aaj ka mausam kaisa hai?",           # → 1 day
        "Aane waale teen din ka mausam bata", # → 3 days
        "Is hafte barish hogi?",              # → 7 days
        "Gehu ka bhav kya hai?",              # → None (not a weather query)
    ]

    for q in test_queries:
        print(f"\n👨‍🌾 Farmer: {q}")
        result = await handle_weather_query(q)
        if result is None:
            print("🤖 [Not a weather query — passed to Mandi/General handler]")
        else:
            print(f"🤖 KisanSaathi:\n{result}")

# In Jupyter:
await test_weather()

# In .py file:
# if __name__ == "__main__":
#     asyncio.run(test_weather())



👨‍🌾 Farmer: Aaj ka mausam kaisa hai?
🤖 KisanSaathi:
🌤️  Mausam Forecast — Yamuna Khadar Field (Agra)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📅 2026-04-24
   🌡️  Temp: 26.2°C – 42.2°C
   ☀️  Baarish nahi
   💨 Hawa: 11.9 km/h

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

👨‍🌾 Farmer: Aane waale teen din ka mausam bata
🤖 KisanSaathi:
🌤️  Mausam Forecast — Yamuna Khadar Field (Agra)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📅 2026-04-24
   🌡️  Temp: 26.2°C – 42.2°C
   ☀️  Baarish nahi
   💨 Hawa: 11.9 km/h

📅 2026-04-25
   🌡️  Temp: 27.5°C – 42.8°C
   ☀️  Baarish nahi
   💨 Hawa: 22.6 km/h

📅 2026-04-26
   🌡️  Temp: 28.0°C – 41.8°C
   ☀️  Baarish nahi
   💨 Hawa: 20.5 km/h

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

👨‍🌾 Farmer: Is hafte barish hogi?
🤖 KisanSaathi:
🌤️  Mausam Forecast — Yamuna Khadar Field (Agra)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📅 2026-04-24
   🌡️  Temp: 26.2°C – 42.2°C
   ☀️  Baarish nahi
   💨 Hawa: 11.9 km/h

📅 2026-04-25
   🌡️  Temp: 27.5°C – 42.8°C
   ☀️  Baarish nahi
   💨 Hawa: 22.6 km/h

📅 2026-04-

## Also Giving The Advices:-

In [3]:
import os, asyncpg, httpx, json
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
OPEN_METEO_URL    = "https://api.open-meteo.com/v1/forecast"
DEFAULT_DAYS      = 3

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,           # slight creativity for natural language
    api_key=os.getenv("GROK_API_KEY"),
)


# ─────────────────────────────────────────────
#  LLM #1 — Intent Detection (what + how many days)
# ─────────────────────────────────────────────
async def extract_weather_intent(user_msg: str) -> dict:
    prompt = f"""Analyze this farmer's message. Respond ONLY with JSON.

Message: "{user_msg}"

Rules:
- is_weather_query = true if farmer asks about weather/mausam/barish/garmi/sardi/aandhi/temp
- days = number of days (aaj=1, kal=2, teen din=3, ek hafte=7, do hafte=14). If not mentioned, set 0.

JSON only, no explanation:
{{"is_weather_query": true/false, "days": 0}}"""

    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    try:
        text = resp.content.strip().replace("```json","").replace("```","").strip()
        result = json.loads(text)
        days = int(result.get("days", 0))
        if days <= 0 or days > 16:
            days = DEFAULT_DAYS
        return {"is_weather_query": bool(result.get("is_weather_query")), "days": days}
    except Exception:
        kw = ["mausam","barish","baarish","weather","garmi","sardi","aandhi","temperature","baarish"]
        return {"is_weather_query": any(k in user_msg.lower() for k in kw), "days": DEFAULT_DAYS}


# ─────────────────────────────────────────────
#  PYTHON — Fetch lat/lon from DB
# ─────────────────────────────────────────────
async def get_farmer_lat_lon(farmer_id: str) -> tuple:
    conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
    try:
        row = await conn.fetchrow(
            """SELECT field_name, city_name, center_lat, center_lon
               FROM farm_fields WHERE farmer_id = $1 LIMIT 1""",
            farmer_id
        )
    finally:
        await conn.close()

    if not row:
        raise ValueError("Aapka khet registered nahi hai.")
    return float(row["center_lat"]), float(row["center_lon"]), row["field_name"], row["city_name"]


# ─────────────────────────────────────────────
#  PYTHON — Call Open-Meteo API
# ─────────────────────────────────────────────
async def fetch_weather_from_api(lat: float, lon: float, days: int) -> dict:
    params = {
        "latitude":       lat,
        "longitude":      lon,
        "daily":          "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,weathercode",
        "forecast_days":  days,
        "timezone":       "Asia/Kolkata",
    }
    async with httpx.AsyncClient() as client:
        resp = await client.get(OPEN_METEO_URL, params=params, timeout=15.0)
    resp.raise_for_status()
    return resp.json()


# ─────────────────────────────────────────────
#  LLM #2 — Analyze weather data + Answer intelligently
# ─────────────────────────────────────────────
async def llm_analyze_weather(
    user_msg: str,
    weather_data: dict,
    field_name: str,
    city: str,
    days: int
) -> str:
    """
    Pass the RAW weather data + farmer's original question to the LLM.
    LLM reads the numbers and gives a smart, contextual Hinglish answer.
    It can answer questions like:
      - "Kya baarish tez hogi?" → looks at precipitation_sum
      - "Gehu ki fasal ke liye kaisa rahega?" → combines temp + rain + wind
      - "Aaj kaam par ja sakta hun?" → looks at today's data
    """

    daily  = weather_data.get("daily", {})
    dates  = daily.get("time", [])
    t_max  = daily.get("temperature_2m_max", [])
    t_min  = daily.get("temperature_2m_min", [])
    rain   = daily.get("precipitation_sum", [])
    wind   = daily.get("windspeed_10m_max", [])

    # Build a clean data summary to feed LLM
    data_lines = []
    for i in range(min(days, len(dates))):
        data_lines.append(
            f"  {dates[i]}: Temp {t_min[i]}–{t_max[i]}°C | "
            f"Baarish {rain[i]}mm | Hawa {wind[i]} km/h"
        )
    weather_summary = "\n".join(data_lines)

    prompt = f"""Tu KisanSaathi hai — ek samajhdar, desi aur helpful agricultural assistant.

Farmer ka khet: {field_name}, {city}
Farmer ka sawaal: "{user_msg}"

Open-Meteo se mila aaj ka ASLI weather data ({days} din ke liye):
{weather_summary}

Iska jawab Hinglish mein de (Hindi + English mix). Tere jawab mein:
1. Farmer ke SAWAL ka seedha jawab de (jaise "Haan, tez baarish aayegi" ya "Nahi, aasman saaf rahega")
2. Numbers ko simple bhasha mein explain kar (jaise "40mm baarish matlab bahut tez baarish")  
3. Fasal ya khet ke liye relevant farming advice bhi de (kya karna chahiye, kya nahi)
4. Warm aur desi tone rakh — jaise gaon ka bada bhai baat kar raha ho
5. Response 4-6 lines mein rakh, zyada lamba mat karna

Bullet points ya headers mat use karna. Natural conversation mein jawab de."""

    response = await llm.ainvoke([HumanMessage(content=prompt)])
    return response.content.strip()


# ─────────────────────────────────────────────
#  MAIN HANDLER
# ─────────────────────────────────────────────
async def handle_weather_query(user_msg: str, farmer_id: str = SESSION_FARMER_ID) -> str | None:
    """
    Returns:
      - LLM-generated intelligent Hinglish string → if weather query
      - None → if NOT a weather query (caller passes to Mandi/General handler)

    PIPELINE:
      LLM #1  → detect intent + extract days       (NLP only)
      Python  → fetch lat/lon from DB               (asyncpg, no LLM)
      Python  → call Open-Meteo API                 (httpx, no LLM)
      LLM #2  → read raw data + answer farmer's Q   (intelligence here!)
    """

    # Step 0 — Is it a weather query?
    intent = await extract_weather_intent(user_msg)
    if not intent["is_weather_query"]:
        return None

    days = intent["days"]

    # Step 1 — Python fetches location (no LLM)
    try:
        lat, lon, field_name, city = await get_farmer_lat_lon(farmer_id)
    except ValueError as e:
        return f"⚠️ {e}"
    except Exception as e:
        return f"⚠️ Database error: {e}"

    # Step 2 — Python calls API (no LLM)
    try:
        weather_data = await fetch_weather_from_api(lat, lon, days)
    except Exception as e:
        return f"⚠️ Mausam server se data nahi aaya: {e}"

    # Step 3 — LLM analyzes data + generates smart answer
    return await llm_analyze_weather(user_msg, weather_data, field_name, city, days)


# ─────────────────────────────────────────────
#  TEST
# ─────────────────────────────────────────────
async def test_weather():
    queries = [
        "Kya aane waale teen din mein tez baarish hogi?",
        "Aaj khet mein kaam kar sakta hun kya?",
        "Is hafte mausam kaisa rahega?",
        "Gehu ki fasal ke liye mausam theek hai?",
        "Gehu ka bhav kya hai?",      # ← should return None
    ]
    for q in queries:
        print(f"\n👨‍🌾 Farmer: {q}")
        result = await handle_weather_query(q)
        if result is None:
            print("🤖 [Not weather — routed to Mandi/General handler]")
        else:
            print(f"🤖 KisanSaathi: {result}")

await test_weather()



👨‍🌾 Farmer: Kya aane waale teen din mein tez baarish hogi?
🤖 KisanSaathi: Arre bhai, aane waale teen din mein tez baarish nahi hogi, kyunki Open-Meteo ka data dekhne se pata chalta hai ki baarish 0.0mm hai. Yeh matlab aasman saaf rahega, toh tum apne khet ki safai aur nayi fasal ki taiyaari kar sakte ho. Abhi tak 40 degree se upar temperature hai, isliye paani ki zaroorat bhi pad sakti hai, toh apne paudhon ko pani dena mat bhoolna. Bas, apne khet ki dekhbhal karte raho, aur agle hafte ka mausam dekhte hain.

👨‍🌾 Farmer: Aaj khet mein kaam kar sakta hun kya?
🤖 KisanSaathi: Arre bhai, aaj khet mein kaam kar sakta hai, kyunki aaj baarish 0.0mm hai, matlab bilkul baarish nahi hai. Temperature 26.2–42.2°C hai, thoda garmi hogi, lekin kaam karne mein koi problem nahi hogi. Hawa 11.9 km/h hai, thoda hawa chalegi, lekin yeh bhi kaam karne mein madad karegi. Toh, aaj khet mein kaam karne ka mauka hai, jaldi shuru karo bhai!

👨‍🌾 Farmer: Is hafte mausam kaisa rahega?
🤖 KisanSaathi: Arre bhai, 